# Experiment 5: data-only baseline
Train without separate initial-condition, boundary-condition, or PDE-residual losses. The optimizer objective contains only supervised data over the complete training grid. Validation and test labels remain held out.

In [ ]:
!pip install -q wandb accelerate

In [ ]:
import os

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
else:
    os.chdir(project_root)
    !git pull

os.environ['PYTHONPATH'] = project_root
%cd {project_root}

from kaggle_secrets import UserSecretsClient
wandb_api_key = UserSecretsClient().get_secret('WANDB_KEY').strip()
if len(wandb_api_key) < 40:
    raise ValueError(
        'Kaggle secret WANDB_KEY is not a valid W&B API key. '
        'Replace it with the key from https://wandb.ai/authorize.'
    )
os.environ['WANDB_API_KEY'] = wandb_api_key

In [ ]:
# This run uses every training-grid label through data_loss. No residual,
# separate IC, or separate BC loss is computed.
FLOW_CASE = 'steep'        # smooth, steep, or shock
SEED = 42

case_presets = {
    'smooth': {
        'nx': 256, 'steps': 200, 't_final': 0.6,
        'mu': 1e-2, 'spatial_scheme': 'central',
    },
    'steep': {
        'nx': 384, 'steps': 250, 't_final': 0.4,
        'mu': 3e-3, 'spatial_scheme': 'rusanov',
    },
    'shock': {
        'nx': 512, 'steps': 300, 't_final': 0.3,
        'mu': 5e-4, 'spatial_scheme': 'rusanov',
    },
}
assert FLOW_CASE in case_presets
case = case_presets[FLOW_CASE]
base_name = f'navier_stokes_{FLOW_CASE}'
data_file = os.path.join(project_root, 'navier-stokes', 'data', base_name + '.pt')

import subprocess
import sys
subprocess.run([
    sys.executable, 'navier-stokes/data/gen_data.py',
    '--save_dir', 'navier-stokes/data',
    '--base_name', base_name,
    '--flow_case', FLOW_CASE,
    '--spatial_scheme', case['spatial_scheme'],
    '--nx', str(case['nx']),
    '--steps', str(case['steps']),
    '--t_final', str(case['t_final']),
    '--mu', str(case['mu']),
], check=True)

In [ ]:
import json

config = {
    'problem': 'navier_stokes',
    'layers': [2, 64, 64, 64, 3],
    'learning_rate': 1e-3,
    'num_steps': 50_000,
    'adapt_every_steps': 100,
    'log_every_steps': 50,
    'batch_size': 8192,
    'seed': SEED,
    'train_time_fraction': 0.7,
    'validation_time_fraction': 0.85,
    'num_data_points': 'all',  # every point with t <= training cutoff
    'loss_terms': ['data'],
    'balancer_type': 'uniform',
    'wandb_project': 'loss-weight-convergence',
    'wandb_run_name': f'navier-stokes-{FLOW_CASE}-data-only-all-seed-{SEED}',
}

os.environ['PINN_CONFIG'] = json.dumps(config)
os.environ['PINN_DATA_FILE'] = data_file

In [ ]:
# One process keeps sampling and reporting deterministic.
!accelerate launch {project_root}/train.py

In [ ]:
from IPython.display import Image, display
plot_name = config['wandb_run_name'] + '_comparison.png'
display(Image(filename=os.path.join(project_root, 'results', plot_name)))
print('Convergence tables and time/step plots are available in the W&B run.')